# Final Model - 최종 모델

## 실험 결과 요약 (전 실험 기반)

| 실험 | 결론 |
|---|---|
| Feature Selection | 전체 feature (stress_level 제외) |
| 정규화 방법 | MinMaxScaler |
| 결측치 처리 | 평균값 대체 |

## 최종 모델 추가 최적화

- Dropout(0.4) 추가 → 과적합 억제
- 클래스 가중치 적용 → 데이터 불균형 보정 (Medium이 많고 Low/High가 적음)
- Early Stopping (patience=15) → 최적 epoch에서 자동 종료
- learning rate 0.001 → 0.003 조정

## 1. 라이브러리 임포트

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

## 2. 데이터 로드 및 전처리

실험 3까지의 결론을 적용: 평균 대체 + MinMaxScaler

In [ ]:
data = pd.read_csv('/content/sample_data/developer_burnout_dataset_7000.csv')
print('데이터 크기:', data.shape)

data['burnout_level'].fillna(data['burnout_level'].mode()[0], inplace=True)
data['burnout_level'] = data['burnout_level'].map({'Low': 0, 'Medium': 1, 'High': 2})

data.fillna(data.mean(numeric_only=True), inplace=True)

print('결측치 처리 후:', data.isnull().sum().sum(), '개')
print('\nburnout_level 분포:')
print(data['burnout_level'].value_counts())

## 3. Feature / Target 정의 및 데이터 분할

In [ ]:
feature_columns = ['age','experience_years','daily_work_hours','sleep_hours',
                   'caffeine_intake','bugs_per_day','commits_per_day',
                   'meetings_per_day','screen_time','exercise_hours']
target_column = ['burnout_level']

X_train, X_test, y_train, y_test = train_test_split(
    data[feature_columns], data[target_column], test_size=0.2, random_state=42)

print(f'Train: {X_train.shape} / Test: {X_test.shape}')

## 4. 정규화 (MinMaxScaler)

In [ ]:
mm = MinMaxScaler()
X_train = mm.fit_transform(X_train)
X_test  = mm.transform(X_test)

## 5. Tensor 변환 및 DataLoader

In [ ]:
X_train = torch.FloatTensor(X_train)
X_test  = torch.FloatTensor(X_test)
y_train = torch.LongTensor(y_train.values)
y_test  = torch.LongTensor(y_test.values)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test,  y_test),  batch_size=64, shuffle=False)

## 6. 모델 정의

베이스라인 대비 변경사항:
- Dropout(0.4) 추가 → 과적합 억제
- 레이어 축소 (4층 → 3층) → 파라미터 수 감소

In [ ]:
class BurnoutModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(32, 3)
        )

    def forward(self, x):
        return self.layer(x)

model = BurnoutModel(10)

## 7. 손실함수 / 옵티마이저

클래스 가중치 적용 이유:
- Medium(1): 3485개로 가장 많음 → 가중치 1.0
- Low(0), High(2): 상대적으로 적음 → 가중치 2.0

In [ ]:
# 클래스 불균형 보정
class_weights = torch.tensor([2.0, 1.0, 2.0])

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.003)

## 8. 학습 (Early Stopping 적용)

In [ ]:
epochs  = 200
patience = 15
best_test_loss = float("inf")
counter = 0
history = {"train_loss": [], "test_loss": [], "train_acc": [], "test_acc": []}

for epoch in range(epochs):
    model.train()
    train_loss, correct, total = 0, 0, 0

    for X, y in train_loader:
        optimizer.zero_grad()
        outputs = model(X)
        y_processed = y.long().squeeze()
        loss = criterion(outputs, y_processed)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        predicted = torch.argmax(outputs, dim=1)
        correct += (predicted == y_processed).sum().item()
        total += y.size(0)

    train_accuracy = correct / total
    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    test_loss, test_correct, test_total = 0, 0, 0

    with torch.no_grad():
        for X, y in test_loader:
            outputs = model(X)
            y_processed = y.long().squeeze()
            loss = criterion(outputs, y_processed)
            test_loss += loss.item()
            predicted = torch.argmax(outputs, dim=1)
            test_correct += (predicted == y_processed).sum().item()
            test_total += y.size(0)

    test_accuracy = test_correct / test_total
    avg_test_loss = test_loss / len(test_loader)

    history["train_loss"].append(avg_train_loss)
    history["test_loss"].append(avg_test_loss)
    history["train_acc"].append(train_accuracy)
    history["test_acc"].append(test_accuracy)

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Test Loss: {avg_test_loss:.4f}, Train Acc: {train_accuracy:.4f}, Test Acc: {test_accuracy:.4f}")

    # Early Stopping
    if avg_test_loss < best_test_loss:
        best_test_loss = avg_test_loss
        counter = 0
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        counter += 1
        if counter >= patience:
            print(f"== Early Stopping triggered at Epoch {epoch+1} ==")
            break

## 9. 학습 결과 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(history['train_loss'], label='Train Loss', color='blue', linestyle='--')
axes[0].plot(history['test_loss'],  label='Test Loss',  color='red')
axes[0].set_title('Final Model - Loss')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history['train_acc'], label='Train Acc', color='blue', linestyle='--')
axes[1].plot(history['test_acc'],  label='Test Acc',  color='red')
axes[1].set_title('Final Model - Accuracy')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"최종 Train Acc: {history['train_acc'][-1]:.4f} | Test Acc: {history['test_acc'][-1]:.4f}")

## 10. 혼동 행렬 및 분류 리포트

In [ ]:
def plot_confusion_matrix(model, test_loader):
    model.eval()
    all_preds, all_targets = [], []

    with torch.no_grad():
        for X, y in test_loader:
            outputs = model(X)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(y.cpu().numpy())

    cm = confusion_matrix(all_targets, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Low', 'Medium', 'High'],
                yticklabels=['Low', 'Medium', 'High'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Burnout Level Prediction - Confusion Matrix')
    plt.show()

    print(classification_report(all_targets, all_preds, target_names=['Low', 'Medium', 'High']))

plot_confusion_matrix(model, test_loader)

## 결론 및 회고

### 성과
- Feature Selection, 정규화, 결측치 처리를 체계적으로 실험해 최적 파이프라인 구축
- Dropout + 클래스 가중치 + Early Stopping 조합으로 과적합 억제

### 한계
- 정확도가 80% 수준에서 정체
- 혼동 행렬 기준 실제 Medium 데이터를 Low/High로 오분류하는 사례 존재
- 클래스 간 경계 데이터의 특징이 유사해 완벽한 분리에 한계

### 향후 과제
- 파생 변수 도입 (10개 feature 조합으로 새로운 feature 생성)
- SMOTE 적용으로 클래스 불균형 물리적 보완